dodac koment....


In [ ]:
import pandas as pd
import numpy as np
import shap
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, precision_score, recall_score,f1_score
import seaborn as sns
from utils import feature_in, feature_in_LR, feature_in_SVC, feature_in_XGB, make_preprocessor

In [ ]:
df = pd.read_csv('Data/Bank Customer Churn Prediction.csv')
customer_ids = df['customer_id']
df = df.drop('customer_id', axis=1)

y = df['churn']
X = df.drop('churn', axis=1)

X_train_raw, X_test_raw, y_train, y_test, id_train, id_test = train_test_split(
    X, y,
    customer_ids,
    test_size=0.2,
    random_state=42,
    stratify=y
)
df_X_train_raw = pd.concat([X_train_raw,y_train], axis=1)
df_to_nn = df_X_train_raw[df_X_train_raw['churn']==0]
df_to_nn.to_csv("data_nn.csv", index=False)


In [ ]:
def make_preprocessor(X, scale_numeric=True):
    num_cols = X.select_dtypes(include=['int64', 'float64']).columns
    cat_cols = X.select_dtypes(include=['object', 'string']).columns
    num_step = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        num_step.append(('scaler', StandardScaler()))
    num_pipeline = Pipeline(num_step)

    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    return ColumnTransformer([
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ])

In [ ]:
#the hyperparameters are obtained using Optuna optimization from SVC_optuna.ipynb
model_SVC = Pipeline([
    ('features', FunctionTransformer(feature_in_SVC)),
    ('preprocessor', make_preprocessor(feature_in_SVC(X_train_raw), scale_numeric=True)),
    ('model', SVC(
        probability=True,
        random_state=42,
        kernel='rbf',
        C=0.308,
        class_weight='balanced',
        gamma='scale'
    ))
])

In [ ]:
#the hyperparameters are obtained using Optuna optimization from LR_optuna.ipynb
model_LR = Pipeline([
    ('features', FunctionTransformer(feature_in_LR)),
    ('preprocessor', make_preprocessor(feature_in_LR(X_train_raw), scale_numeric=True)),
    ('model', LogisticRegression(
        solver='lbfgs',
        C=3.7,
        max_iter=2541,
        class_weight='balanced',
        random_state=42
    ))
])

In [ ]:
#the hyperparameters are obtained using Optuna optimization from XGB_optuna.ipynb
model_XGB = Pipeline([
    ('features', FunctionTransformer(feature_in_XGB)),
    ('preprocessor', make_preprocessor(feature_in_XGB(X_train_raw), scale_numeric=False)),
    ('model', XGBClassifier(
        n_estimators=1067,
        max_depth=6,
        learning_rate=0.082,
        subsample=0.846,
        colsample_bytree=0.6275,
        gamma=4.58,
        min_child_weight=6,
        reg_alpha=4.832,
        reg_lambda=0.1434,
        scale_pos_weight=2.92,
        eval_metric='logloss'
    ))
])

In [ ]:
voting_model = VotingClassifier(
    estimators=[
        ('svc', model_SVC),
        ('lr', model_LR),
        ('xgb', model_XGB)
    ],
    voting='soft',
    weights=[1, 1, 1],
    n_jobs=-1
)

In [ ]:
# cross valu score precision, recall, roc_auc
for test in ["precision", "recall", "roc_auc"]:
    punkty_model_LR=cross_val_score(estimator=voting_model,X=X_train_raw,y=y_train, cv=5,scoring=test)
    print(test,": ", punkty_model_LR.mean().round(2))

In [ ]:
voting_model.fit(X_train_raw, y_train)

In [ ]:
import joblib
joblib.dump(voting_model, 'models/voting_model.pkl')

In [ ]:
y_pred = voting_model.predict(X_test_raw)
y_proba = voting_model.predict_proba(X_test_raw)[:, 1]

In [ ]:
y_pred

In [ ]:
y_proba

In [ ]:
model_XGB.fit(X_train_raw, y_train)

preproc = model_XGB.named_steps['preprocessor']
XGB_model = model_XGB.named_steps['model']

X_train_feat = feature_in_XGB(X_train_raw)
X_test_feat = feature_in_XGB(X_test_raw)

X_train_t = preproc.transform(X_train_feat)
X_test_t = preproc.transform(X_test_feat)

feature_names = preproc.get_feature_names_out()
feature_names = [k.split('__')[1] for k in feature_names]
explainer = shap.TreeExplainer(XGB_model)

shap_values = explainer(X_test_t)

shap_values

In [ ]:
# Na samym końcu tego bloku, który wkleiłeś:
joblib.dump(model_XGB, 'models/xgboost_for_shap.pkl')

In [ ]:
shap.summary_plot(shap_values, X_test_t, feature_names=feature_names)

In [ ]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[0,30,40,50,100]
)

df.groupby('age_group')['churn'].mean()

In [ ]:

feature_names_waterf = preproc.get_feature_names_out()
X_test_t_df = pd.DataFrame(
    X_test_t,
    columns=[k.split('__')[1] for k in feature_names_waterf]
)
shap_values = explainer(X_test_t_df)

In [ ]:
feature_names_waterf

In [ ]:
# shap data frame for final file
df_shap = pd.DataFrame(shap_values.values, columns=[f"shap_{col}" for col in feature_names], index=X_test_raw.index)
df_shap.iloc[1]

In [ ]:
df_shap

In [ ]:
# Precision/Recall as a function of the decision threshold

precisions = []
recalls = []
f1_scores = []
thresholds = np.linspace(0, 1, 100)

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)

    precisions.append(round(precision_score(y_test, y_pred_t, zero_division=1), 3))
    recalls.append(round(recall_score(y_test, y_pred_t, zero_division=0), 3))
    f1_scores.append(round(f1_score(y_test, y_pred_t, zero_division=0), 3))

# data frame for searching decisions thresholds
df_scores_t = pd.DataFrame({
    "threshold": thresholds,
    "f1": f1_scores,
    "precision": precisions,
    "recall": recalls
})

# plt.figure(figsize=(10, 6))
#
# plt.plot(thresholds, precisions, label='Precision')
# plt.plot(thresholds, recalls, label='Recall')
#
# plt.xlabel('Decision Threshold')
# plt.ylabel('Score')
# plt.title('Precision and Recall vs Decision Threshold')
#
# plt.grid(True)
# plt.legend()
#
# plt.show()

In [ ]:
df_scores_t

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(thresholds, precisions, label='Precision')
plt.plot(thresholds, recalls, label='Recall')

plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title('Precision and Recall vs Decision Threshold')

plt.grid(True)
plt.legend()

plt.show()

In [ ]:
# serch decision threshold for max f1 and f1 max
t_f1_max = df_scores_t.loc[df_scores_t['f1'].idxmax(), 'threshold']
best_f1 = df_scores_t.loc[df_scores_t['f1'].idxmax(), 'f1']

# serch decision threshold for precision bigger then 0.89

mask_p90 = (df_scores_t['precision'] > 0.89)
t_prec_90_row = df_scores_t.loc[
    df_scores_t.loc[mask_p90, 'threshold'].idxmin()]

t_prec_90 = t_prec_90_row['threshold']

print(f'Threshold: {t_f1_max.round(2)},for best F1: {best_f1}')
print(f'Precision = 0.9 , for threshold: {t_prec_90.round(2)}')

In [ ]:
# flag for increased marketing
y_pred_mark_a = (y_proba >= t_prec_90).astype(int)
conf_matrix_90 = confusion_matrix(y_test, y_pred_mark_a)
print(conf_matrix_90)

In [ ]:
y_pred_mark_a

In [ ]:
# flag for normal marketing
y_pred_mark_n = (y_proba >= t_f1_max).astype(int)
conf_matrix_f1 = confusion_matrix(y_test, y_pred_mark_n)
print(conf_matrix_f1)

In [ ]:
# the file for the next analysis in LangChain.
df_marketing = pd.DataFrame({
    'churn': y_pred_mark_a},
    index=X_test_raw.index,

)
df_final = pd.concat(
    [X_test_raw, df_shap, df_marketing],
    axis=1)
df_final['customer_id'] = id_test.values

In [ ]:
# print(len(X_test_raw))
# print(len(df_marketing))
# print(len(df_final))


In [ ]:

df_final.to_csv('bank_c_marketing.csv', index=False)

In [ ]:
!jupyter nbconvert --to html Final_Vote_model_LC.ipynb